In [1]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics as qa
client = StockClient(ttl_price=60, ttl_info=900)
quant = qa(client)

In [3]:
from yfinance_api3.modules.factors import run, run_bulk, compare_models, rolling_betas

In [4]:
# single stock, single model
result = run(quant, "AAPL", model="ff5", period="5y")
print(result)

FactorResult — AAPL  [FF5]  5y
  Alpha (ann.)  : +2.982%   (p=0.711)
  R²            : 0.608   Adj-R²: 0.606   N: 1216

  Factor       Beta   t-stat    p-val
  --------------------------------------
  Mkt-RF      1.222    38.16    0.000 ***
  SMB        -0.069    -1.28    0.200 
  HML        -0.396    -8.10    0.000 ***
  RMW         0.516     8.61    0.000 ***
  CMA         0.269     3.93    0.000 ***


In [5]:
# all four models side by side
df = compare_models(quant, "AAPL", period="5y")
df

,alpha,alpha_pval,r_squared,adj_r2,n_obs
FF3,0.040699,0.626894,0.578164,0.577120,1216.0
FF5,0.029815,0.710819,0.607529,0.605907,1216.0
MOM,0.046680,0.575279,0.585285,0.583915,1216.0
FF6,0.037816,0.636323,0.615322,0.613413,1216.0


In [6]:
# entire basket at once
df = run_bulk(quant, ["AAPL","MSFT","NVDA"], model="ff5", period="3y")
df

,alpha,r_squared,Mkt-RF,SMB,HML,RMW,CMA
AAPL,-0.023273,0.483297,1.220076,-0.001291,-0.264282,0.704367,0.107194
MSFT,-0.080013,0.544324,0.954517,-0.238097,-0.400504,0.193595,-0.632935
NVDA,0.554845,0.540696,2.030943,-0.607548,-1.257814,0.455205,0.277217


In [7]:
# how exposures shift over time
df = rolling_betas(quant, "AAPL", model="ff3", window=126)
df

,alpha_daily,Mkt-RF,SMB,HML
2021-10-22,-0.000027,1.376620,-0.474660,-0.253698
2021-10-25,-0.000070,1.373574,-0.479714,-0.252167
2021-10-26,-0.000041,1.372268,-0.477657,-0.251785
2021-10-27,0.000018,1.382599,-0.501508,-0.243318
2021-10-28,0.000176,1.388480,-0.486202,-0.251665
...,...,...,...,...
2026-02-23,0.000685,0.969760,-0.366589,0.171644
2026-02-24,0.000821,0.977659,-0.355820,0.153119
2026-02-25,0.000818,0.971430,-0.352959,0.150325
2026-02-26,0.000787,0.967875,-0.352945,0.149109


Reading the output:

alpha > 0 and significant → the stock generated returns beyond what the factors explain
High Mkt-RF beta → amplifies market moves
Positive SMB → behaves like a small-cap even if it isn't
Positive HML → value tilt; negative → growth tilt
High R² → most of the stock's variance is explained by systematic factors; low R² → more idiosyncratic

In [8]:
import yfinance_api3.modules.plots as plots

In [9]:
plots.factor_exposure(result)

In [10]:
plots.rolling_factor_betas(quant, 'AAPL')